# QSS Trading System - Master Workflow

This notebook provides an interactive interface for the complete QSS trading system workflow:
- **Section 1:** Data Curator - Manage ticker universe and data collection
- **Section 2:** SEPA Model - Build datasets and train ML models
- **Section 3:** Daily Scanner - Run scans and manage buy list

In [1]:
# ============================================================================
# SETUP: Import Dependencies
# ============================================================================

import sys
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta
import json
import warnings
warnings.filterwarnings('ignore')

# IPython display utilities
from IPython.display import display, HTML
from tqdm import tqdm

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# Add project root to path
project_root = Path.cwd()
sys.path.append(str(project_root))

# Force reload environment variables (important if .env was created after kernel started)
from dotenv import load_dotenv
load_dotenv(override=True)

# Import project modules
import config
from src.data_engine import DataRepository
from src.database import DatabaseManager

print("✅ All dependencies loaded successfully!")
print(f"📁 Project root: {project_root}")
print(f"🔧 Config loaded from: {config.__file__}")

# Verify API key is loaded
if config.FMP_API_KEY:
    print(f"🔑 FMP API Key: {config.FMP_API_KEY[:8]}... (loaded)")
else:
    print("⚠️  WARNING: FMP API Key not found in .env file!")

✅ All dependencies loaded successfully!
📁 Project root: /Users/ceolwaerc/Desktop/Projects/mm-strat
🔧 Config loaded from: /Users/ceolwaerc/Desktop/Projects/mm-strat/config.py
🔑 FMP API Key: ijhOTOkH... (loaded)


---
# SECTION 1: DATA CURATOR
---

## Cell 1: Get Tickers

Load ticker universe from three possible sources:
1. **Price Folder** - All tickers with cached price data
2. **FMP Screener** - Filtered universe from Financial Modeling Prep API
3. **S&P 500** - Standard & Poor's 500 index constituents

In [3]:
# ============================================================================
# CELL 1: Get Tickers
# ============================================================================

print("📊 TICKER UNIVERSE LOADER")
print("=" * 80)

# === CONFIGURATION ===
# Purpose: Define which tickers to work with for data collection and analysis
# Options: 'price_folder' (use existing cached tickers), 'fmp_screener' (fetch from FMP API), 'sp500' (S&P 500 constituents)
ticker_source = 'price_folder'  # Example: 'price_folder', 'fmp_screener', 'sp500'

# FMP Screener Parameters (only used if ticker_source = 'fmp_screener')
# Purpose: Filter stocks based on fundamental criteria
market_cap_min = 300000000  # Example: 1000000000 (1B market cap minimum)
price_min = 5.0              # Example: 5.0 (stocks above $5)
volume_min = 100000          # Example: 100000 (minimum daily volume)

# Custom Tickers (optional - overrides ticker_source if provided)
# Purpose: Manually specify exact tickers to analyze
# Example: 'AAPL,MSFT,TSLA,NVDA,META' or leave empty to use ticker_source
custom_tickers_input = ''

# === EXECUTION ===
loaded_tickers = []  # Global variable to store tickers

try:
    print(f"\n🔄 Loading tickers from source: {ticker_source}...\n")

    # Check if custom tickers provided
    if custom_tickers_input.strip():
        loaded_tickers = [t.strip().upper() for t in custom_tickers_input.split(',') if t.strip()]
        print(f"✅ Using {len(loaded_tickers)} custom tickers")
        print(f"   Sample: {', '.join(loaded_tickers[:10])}{'...' if len(loaded_tickers) > 10 else ''}")
    else:
        # Initialize data repository
        data_repo = DataRepository()

        # Map UI selection to source parameter
        source_map = {
            'price_folder': 'PRICE_FOLDER',
            'fmp_screener': 'FMP_SCREENER',
            'sp500': 'SSGA'
        }
        source = source_map[ticker_source]

        # Update FMP screener params if needed
        if ticker_source == 'fmp_screener':
            config.FMP_SCREENER_PARAMS.update({
                'marketCapMoreThan': market_cap_min,
                'priceMoreThan': price_min,
                'volumeMoreThan': volume_min
            })
            print(f"   Using screener filters:")
            print(f"     Market Cap > ${market_cap_min:,}")
            print(f"     Price > ${price_min}")
            print(f"     Volume > {volume_min:,}")

        # Load tickers
        loaded_tickers = data_repo.update_universe(source=source)

        # Display results
        print(f"\n✅ Successfully loaded {len(loaded_tickers)} tickers")
        print(f"\n📋 Sample tickers (first 20):")
        print(f"   {', '.join(sorted(loaded_tickers[:20]))}")
        if len(loaded_tickers) > 20:
            print(f"   ... and {len(loaded_tickers) - 20} more")

        # Display statistics
        print(f"\n📊 Statistics:")
        print(f"   Total tickers: {len(loaded_tickers)}")
        print(f"   Source: {ticker_source}")

except Exception as e:
    print(f"❌ Error loading tickers: {e}")
    import traceback
    traceback.print_exc()

INFO:src.data_engine:Fetching ticker universe from source: PRICE_FOLDER
INFO:src.data_engine:Scanning data/price folder for tickers...
INFO:src.data_engine:Found 2413 tickers from price folder


📊 TICKER UNIVERSE LOADER

🔄 Loading tickers from source: price_folder...


✅ Successfully loaded 2413 tickers

📋 Sample tickers (first 20):
   ABL, ALKT, APGE, BDJ, BIT, BOC, BTZ, BYD, CBRE, CCB, CSR, FORM, IAS, IVT, MTH, OTF, SGRY, SITC, SUI, TRUP
   ... and 2393 more

📊 Statistics:
   Total tickers: 2413
   Source: price_folder


## Cell 2: Price Cache Update

Initialize or update price data cache using batch API calls.

In [10]:
# ============================================================================
# CELL 2: Price Cache Update
# ============================================================================

print("📈 PRICE DATA CACHE UPDATER")
print("=" * 80)

# === CONFIGURATION ===
# Purpose: Download and cache historical price data for tickers
use_loaded_tickers_2 = True  # Use tickers from Cell 1 (True) or custom list (False)

# Data Source
# Purpose: Choose API provider for price data
# Options: 'fmp' (Financial Modeling Prep - parallel downloads), 'yfinance' (Yahoo Finance - batch downloads)
price_source = 'fmp'  # Example: 'fmp', 'yfinance'

# Force Update
# Purpose: Control cache update behavior
# - False (default): Only download missing tickers or tickers missing latest trading day
#                    SKIPS update during market hours (9:30am-4pm ET) to avoid incomplete data
# - True: Force re-download ALL tickers regardless of cache (overrides market hours check)
# Best practice: Run after 4:00 PM ET for complete closing prices
force_update_price = False  # Example: False (incremental), True (full refresh)

# Parallel Processing (FMP only)
# Purpose: Number of concurrent workers for downloading (maximizes API rate limit usage)
# Recommended: 10 workers for FMP Starter tier (300 calls/min rate limit)
max_workers_price = 10  # Example: 10 (FMP), not used for yfinance (uses built-in batching)

# === EXECUTION ===
try:
    # Get tickers
    if use_loaded_tickers_2:
        if not loaded_tickers:
            print("❌ No tickers loaded! Please run Cell 1 first.")
            raise ValueError("No tickers available")
        tickers = loaded_tickers
    else:
        print("❌ Custom ticker selection not implemented. Use Cell 1 tickers.")
        raise ValueError("Custom ticker selection not supported")

    print(f"\n🔄 Updating price cache for {len(tickers)} tickers...")
    print(f"   Source: {price_source}")
    print(f"   Force Update: {force_update_price}")
    
    # Check if during market hours
    from datetime import datetime, time as dt_time
    import pytz
    et_tz = pytz.timezone('US/Eastern')
    now_et = datetime.now(et_tz)
    market_open = dt_time(9, 30)
    market_close = dt_time(16, 0)
    if market_open <= now_et.time() < market_close and not force_update_price:
        print(f"   ⚠️  Market currently open - update will be skipped (run after 4pm ET)")
    
    if price_source == 'fmp':
        print(f"   Parallel Workers: {max_workers_price}")
    print()

    # Initialize data repository
    data_repo = DataRepository()

    # Update cache
    import time
    start_time = time.time()

    # For FMP, pass max_workers parameter
    if price_source == 'fmp':
        # Monkey-patch to pass max_workers
        original_update_cache_fmp = data_repo._update_cache_fmp
        data_repo._update_cache_fmp = lambda tickers: original_update_cache_fmp(tickers, max_workers=max_workers_price, show_progress=True)

    results = data_repo.update_cache(
        tickers,
        force=force_update_price,
        source=price_source
    )

    elapsed_time = time.time() - start_time

    # Count successes
    success_count = sum(results.values())
    fail_count = len(results) - success_count

    # Display results (detailed summary already printed by _update_cache_fmp)
    if price_source != 'fmp':  # FMP already prints detailed summary
        print(f"\n✅ Cache update complete!")
        print(f"\n📊 Results:")
        print(f"   Successful: {success_count}/{len(tickers)} ({success_count/len(tickers)*100:.1f}%)")
        print(f"   Failed: {fail_count}")
        print(f"   Time: {elapsed_time:.1f}s ({len(tickers)/elapsed_time:.1f} tickers/sec)")

        # Show failed tickers if any
        if fail_count > 0:
            failed_tickers = [t for t, success in results.items() if not success]
            print(f"\n⚠️  Failed tickers ({len(failed_tickers)}):")
            print(f"   {', '.join(failed_tickers[:20])}")
            if len(failed_tickers) > 20:
                print(f"   ... and {len(failed_tickers) - 20} more")

except Exception as e:
    print(f"❌ Error updating price cache: {e}")
    import traceback
    traceback.print_exc()

INFO:src.data_engine:Updating cache for 2384/2384 tickers...
INFO:src.data_engine:Using FMP API for batch update...
INFO:src.data_engine:Using yfinance for batch update...
INFO:src.data_engine:Processing batch 1/48


📈 PRICE DATA CACHE UPDATER

🔄 Updating price cache for 2384 tickers...
   Source: fmp
   Min Date: 1990-01-01
   Force Update: True
   Parallel Workers: 10



INFO:src.data_engine:Processing batch 2/48
INFO:src.data_engine:Processing batch 3/48
INFO:src.data_engine:Processing batch 4/48
INFO:src.data_engine:Processing batch 5/48
INFO:src.data_engine:Processing batch 6/48
INFO:src.data_engine:Processing batch 7/48
INFO:src.data_engine:Processing batch 8/48
INFO:src.data_engine:Processing batch 9/48
INFO:src.data_engine:Processing batch 10/48
INFO:src.data_engine:Processing batch 11/48
INFO:src.data_engine:Processing batch 12/48
INFO:src.data_engine:Processing batch 13/48
INFO:src.data_engine:Processing batch 14/48
INFO:src.data_engine:Processing batch 15/48
INFO:src.data_engine:Processing batch 16/48
INFO:src.data_engine:Processing batch 17/48
INFO:src.data_engine:Processing batch 18/48
INFO:src.data_engine:Processing batch 19/48
INFO:src.data_engine:Processing batch 20/48
INFO:src.data_engine:Processing batch 21/48
INFO:src.data_engine:Processing batch 22/48
INFO:src.data_engine:Processing batch 23/48
INFO:src.data_engine:Processing batch 24

## Cell 3: Fundamental Data Update

Initialize or update fundamental data (financial statements, ratios, growth metrics).

In [ ]:
# ============================================================================
# CELL 3: Fundamental Data Update
# ============================================================================

print("📊 FUNDAMENTAL DATA UPDATER")
print("=" * 80)

# === CONFIGURATION ===
# Purpose: Download and cache quarterly fundamental data (income statement, balance sheet)
# Note: Only checks for MISSING tickers, not stale data (fundamentals update quarterly on earnings dates)
# Use force_update_fund=True to re-download existing data

use_loaded_tickers_3 = True  # Use tickers from Cell 1

# Force Update
# Purpose: Re-download ALL tickers (cached and missing) - use only when you want to refresh existing data
force_update_fund = False  # Example: False (incremental), True (full refresh)

# Parallel Processing
# Purpose: Control how many API requests to make concurrently
# Recommended: 10 workers for FMP Starter tier (300 calls/min, 3 calls per ticker = ~100 tickers/min)
parallel_workers_fund = 10  # Example: 10 (recommended), 5 (conservative)

# === EXECUTION ===
try:
    # Get tickers
    if use_loaded_tickers_3:
        if not loaded_tickers:
            print("❌ No tickers loaded! Please run Cell 1 first.")
            raise ValueError("No tickers available")
        tickers = loaded_tickers
    else:
        print("❌ Custom ticker selection not implemented. Use Cell 1 tickers.")
        raise ValueError("Custom ticker selection not supported")

    print(f"
🔄 Updating fundamental data for {len(tickers)} tickers...")
    print(f"   Parallel Workers: {parallel_workers_fund}")
    print(f"   Force Update: {force_update_fund}")
    if force_update_fund:
        print(f"   Estimated API calls: {len(tickers) * 3} (~{len(tickers) * 3 / 300:.1f} minutes at 300 calls/min)")
    else:
        print(f"   Mode: Incremental (only missing tickers will be downloaded)")
    print()

    # Import fundamental engine
    from src.fundamental_engine import FundamentalEngine

    # Initialize engine
    fund_engine = FundamentalEngine()

    # Update fundamentals using the proper batch method with parallel processing
    import time
    start_time = time.time()
    
    results = fund_engine.update_fundamentals_cache(
        tickers=tickers,
        force=force_update_fund,
        show_progress=True,
        max_workers=parallel_workers_fund
    )
    
    elapsed_time = time.time() - start_time

    # Count results
    success_count = sum(results.values())
    fail_count = len(results) - success_count
    failed_tickers = [t for t, success in results.items() if not success]

    # Display results
    print(f"
✅ Fundamental data update complete!")
    print(f"
📊 Results:")
    print(f"   Successful: {success_count}/{len(tickers)} ({success_count/len(tickers)*100:.1f}%)")
    print(f"   Failed: {fail_count}")
    print(f"   Time: {elapsed_time:.1f}s ({elapsed_time/60:.1f} min)")
    
    if failed_tickers:
        print(f"
⚠️  Failed tickers ({len(failed_tickers)}):")
        print(f"   {', '.join(failed_tickers[:20])}")
        if len(failed_tickers) > 20:
            print(f"   ... and {len(failed_tickers) - 20} more")

except Exception as e:
    print(f"❌ Error updating fundamentals: {e}")
    import traceback
    traceback.print_exc()

## Cell 4: Company Profile Update

Initialize or update company profiles (sector, industry, market cap, etc.).

In [ ]:
# ============================================================================# CELL 4: Company Profile Update# ============================================================================print("🏢 COMPANY PROFILE UPDATER")print("=" * 80)# === CONFIGURATION ===# Purpose: Download and cache company profile data (sector, industry, market cap, etc.)# Note: Only checks for MISSING tickers, not stale data (company profiles rarely change)# Use force_update_profile=True to re-download existing profilesuse_loaded_tickers_4 = True  # Use tickers from Cell 1# Force Update# Purpose: Re-download ALL tickers (cached and missing) - use only when profiles have changedforce_update_profile = False  # Example: False (use cache), True (re-download everything)# Parallel Processing# Purpose: Number of concurrent workers for downloading# Recommended: 10 workers for FMP Starter tier (300 calls/min rate limit)max_workers_profile = 10  # Example: 10 (recommended), 5 (conservative)# === EXECUTION ===try:    # Get tickers    if use_loaded_tickers_4:        if not loaded_tickers:            print("❌ No tickers loaded! Please run Cell 1 first.")            raise ValueError("No tickers available")        tickers = loaded_tickers    else:        print("❌ Custom ticker selection not implemented. Use Cell 1 tickers.")        raise ValueError("Custom ticker selection not supported")    print(f"🔄 Updating company profiles for {len(tickers)} tickers...")    print(f"   Force Update: {force_update_profile}")    print(f"   Parallel Workers: {max_workers_profile}")    if force_update_profile:        print(f"   Estimated API calls: {len(tickers)} (~{len(tickers) / 300:.1f} minutes at 300 calls/min)")    else:        print(f"   Mode: Incremental (only missing tickers will be downloaded)")    print()    # Import company profile engine    from src.company_profile_engine import CompanyProfileEngine    # Initialize engine    profile_engine = CompanyProfileEngine()    # Update profiles using the proper batch method    import time    start_time = time.time()        results = profile_engine.update_profiles_cache(        tickers=tickers,        force=force_update_profile,        max_workers=max_workers_profile    )        elapsed_time = time.time() - start_time    # Count results    success_count = sum(results.values())    fail_count = len(results) - success_count    failed_tickers = [t for t, success in results.items() if not success]        # Display results    print(f"✅ Company profile update complete!")    print(f"📊 Results:")    print(f"   Successful: {success_count}/{len(tickers)} ({success_count/len(tickers)*100:.1f}%)")    print(f"   Failed: {fail_count}")    print(f"   Time: {elapsed_time:.1f}s ({elapsed_time/60:.1f} min)")        if failed_tickers:        print(f"⚠️  Failed tickers ({len(failed_tickers)}):")        print(f"   {', '.join(failed_tickers[:20])}")        if len(failed_tickers) > 20:            print(f"   ... and {len(failed_tickers) - 20} more")except Exception as e:    print(f"❌ Error updating profiles: {e}")    import traceback    traceback.print_exc()

## Cell 5: Data Health Check

Comprehensive analysis of data coverage and quality across all dimensions.

In [ ]:
# ============================================================================
# CELL 5: Data Health Check
# ============================================================================

print("🏥 DATA HEALTH ANALYZER")
print("=" * 80)

# === CONFIGURATION ===
# Purpose: Analyze data quality and coverage across all dimensions (price, fundamentals, profiles)

# Save Report
# Purpose: Export detailed health report to JSON file for later review
save_health_report = True  # Example: True (save report), False (display only)

# Report Path
# Purpose: Location to save the detailed analysis report
health_report_path = 'data/data_health_report.json'  # Example: 'data/data_health_report.json'

# === EXECUTION ===
try:
    print(f"\n🔍 Running comprehensive data health analysis...\n")

    # Import data health analyzer
    from data_health_analyzer import DataHealthAnalyzer

    # Run full analysis
    analyzer = DataHealthAnalyzer()
    results = analyzer.run_full_analysis()

    # Save detailed report if requested
    if save_health_report:
        analyzer.save_detailed_report(health_report_path)
        print(f"\n💾 Detailed report saved to: {health_report_path}")

except Exception as e:
    print(f"❌ Error running health check: {e}")
    import traceback
    traceback.print_exc()

---
# SECTION 2: SEPA MODEL
---

## Cell 6: Build Dataset A

Generate daily feature snapshots (technical indicators + optional fundamentals).

In [8]:
# ============================================================================
# CELL 6: Build Dataset A
# ============================================================================

print("📊 DATASET A BUILDER - Daily Feature Snapshots")
print("=" * 80)

# === CONFIGURATION ===
# Purpose: Generate daily feature snapshots combining technical indicators and optional fundamentals

# Date Range
# Purpose: Define time period for dataset generation
start_date_a = '2003-01-01'  # Example: '2020-01-01' (start from this date)
end_date_a = datetime.now().strftime('%Y-%m-%d')  # Example: datetime.now() (up to today)

# Mode
# Purpose: Control feature complexity - lightweight (technical only) or full (technical + fundamentals)
# Options: 'lightweight' (faster, technical indicators only), 'full' (slower, includes fundamentals)
mode_a = 'full'  # Example: 'lightweight', 'full'

# Ticker Selection
# Purpose: Use tickers from Cell 1 or full universe
use_loaded_tickers_a = True  # Example: True (use Cell 1 tickers), False (use all tickers from FMP)

# Feature Groups
# Purpose: Control which feature categories to include
include_fundamentals_a = True  # Include fundamental ratios (P/E, ROE, etc.)
include_cross_sectional_a = True  # Include cross-sectional rankings (percentiles vs universe)

# Performance
# Purpose: Control parallel processing
# -1 = use all CPU cores, 1 = single-threaded, >1 = specific number of cores
n_jobs_a = 6  # Example: 1 (single-threaded), -1 (all cores), 4 (4 cores)

# Data Updates
# Purpose: Skip API calls and use existing cached data
skip_data_updates_a = True  # Example: True (faster, use cache), False (update from API first)

# Output
# Purpose: Where to save the generated dataset
output_path_a = 'data/ml/dataset_a.parquet'  # Example: 'data/ml/dataset_a.parquet'

# === EXECUTION ===
dataset_a = None  # Global variable to store dataset

try:
    print(f"\n🚀 Building Dataset A...\n")

    # Determine tickers
    tickers_to_use = None
    if use_loaded_tickers_a:
        if loaded_tickers:
            tickers_to_use = loaded_tickers
            print(f"   Using {len(tickers_to_use)} tickers from Cell 1")
        else:
            print(f"   ⚠️  No loaded tickers, using full universe")
    else:
        print(f"   Using full universe from FMP screener")

    # Import and run builder
    from build_dataset_a import build_dataset_a

    dataset_a = build_dataset_a(
        start_date=start_date_a,
        end_date=end_date_a,
        mode=mode_a,
        tickers=tickers_to_use,
        validate_temporal=True,
        include_fundamentals=include_fundamentals_a,
        include_cross_sectional=include_cross_sectional_a,
        n_jobs=n_jobs_a,
        skip_data_updates=skip_data_updates_a
    )

    if not dataset_a.empty:
        # Save dataset
        output_path = Path(output_path_a)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        dataset_a.to_parquet(output_path, index=False, compression='snappy')

        # Display summary
        print(f"\n✅ Dataset A built successfully!")
        print(f"\n📊 Summary:")
        print(f"   Total rows: {len(dataset_a):,}")
        print(f"   Features: {len(dataset_a.columns) - 2}")  # Exclude date, ticker
        print(f"   Tickers: {dataset_a['ticker'].nunique()}")
        print(f"   Date range: {dataset_a['date'].min()} to {dataset_a['date'].max()}")
        print(f"   File: {output_path}")
        print(f"   Size: {output_path.stat().st_size / (1024*1024):.2f} MB")
    else:
        print("❌ No data generated!")

except Exception as e:
    print(f"❌ Error building Dataset A: {e}")
    import traceback
    traceback.print_exc()

INFO:build_dataset_a:Building Dataset A from 2003-01-01 to 2025-12-07
INFO:build_dataset_a:Mode: full
INFO:build_dataset_a:Parallel jobs: 6
INFO:src.features:FeatureEngineer initialized in dual-stage mode
INFO:build_dataset_a:Fundamental enrichment enabled
INFO:build_dataset_a:Cache-only mode enabled for fundamentals (no API updates)
INFO:build_dataset_a:Processing 504 tickers
INFO:build_dataset_a:Skipping price data cache update (using existing cache)
INFO:build_dataset_a:Loading ticker data...


📊 DATASET A BUILDER - Daily Feature Snapshots

🚀 Building Dataset A...

   Using 504 tickers from Cell 1


Loading price data:  66%|██████▌   | 333/504 [00:42<00:21,  7.95ticker/s]WARNING:src.data_engine:Unexpected FMP response format for -: <class 'list'>
ERROR:src.data_engine:Failed to download - from FMP after 2 attempts
Loading price data: 100%|██████████| 504/504 [01:03<00:00,  7.92ticker/s]
INFO:build_dataset_a:Successfully loaded 503/504 tickers
INFO:build_dataset_a:Processing 5983 trading days
INFO:build_dataset_a:Processing tickers in parallel using 6 workers...
INFO:build_dataset_a:Using chunksize=10 for 503 tickers
Building Dataset A: 100%|██████████| 503/503 [03:14<00:00,  2.59ticker/s]
INFO:build_dataset_a:Concatenating results...
INFO:build_dataset_a:
INFO:build_dataset_a: APPLYING LAGGED FEATURES (Setup Conditions at T-1)
INFO:build_dataset_a:================================================================================
INFO:build_dataset_a:Creating lagged features with groupby to prevent cross-ticker contamination...
INFO:build_dataset_a:Created 14/14 lagged features in 6.

: 

## Cell 7: Build Dataset B

Generate trade simulation labels using historical backtesting.

In [ ]:
# ============================================================================
# CELL 7: Build Dataset B
# ============================================================================

print("🎯 DATASET B BUILDER - Trade Simulation Labels")
print("=" * 80)

# === CONFIGURATION ===
# Purpose: Generate trade outcomes by simulating historical trades to create ML labels

# Date Range for Entry Signals
# Purpose: Define when trades can be entered
start_date_b = '2020-01-01'  # Example: '2020-01-01' (earliest entry date)
end_date_b = datetime.now().strftime('%Y-%m-%d')  # Example: datetime.now() (latest entry date)

# Outcome Window
# Purpose: How far into the future to track trade outcomes
outcome_end_b = (datetime.now() + timedelta(days=90)).strftime('%Y-%m-%d')  # Example: 90 days from now

# Success Criteria
# Purpose: Minimum return % to classify a trade as "successful" (label=1)
success_threshold_b = 15.0  # Example: 15.0 (15% return required for success)

# Simulator Type
# Purpose: Choose between fast vectorized or event-driven simulator
# Fast vectorized is 10-20x faster but uses more memory
use_fast_simulator_b = True  # Example: True (vectorized, faster), False (event-driven, slower)

# Performance
# Purpose: Control parallel processing
n_jobs_b = 1  # Example: 1 (single-threaded), -1 (all cores)

# Custom Labeling Rule (optional)
# Purpose: Define custom success criteria beyond just return threshold
# Example: 'trade.return_pct >= 20 and trade.days_held <= 30' (20% in under 30 days)
# Leave empty to use default threshold-based labeling
custom_label_rule_b = ''  # Example: '', 'trade.return_pct >= 20 and trade.days_held <= 30'

# Output
# Purpose: Where to save the generated dataset
output_path_b = 'data/ml/dataset_b.parquet'  # Example: 'data/ml/dataset_b.parquet'

# === EXECUTION ===
dataset_b = None  # Global variable to store dataset

try:
    print(f"\n🚀 Building Dataset B (Trade Simulation)...\n")
    print(f"   Entry Period: {start_date_b} to {end_date_b}")
    print(f"   Outcome Window: {outcome_end_b}")
    print(f"   Success Threshold: {success_threshold_b}%")
    print(f"   Simulator: {'Fast Vectorized' if use_fast_simulator_b else 'Event-Driven'}\n")

    # Import components
    from src.data_engine import DataRepository
    from src.strategy import SEPAStrategy
    from src.features import FeatureEngineer
    from src.trading_config import TradingConfig

    # Initialize components
    data_repo = DataRepository()
    benchmark_data = data_repo.get_benchmark_data()
    feature_engine = FeatureEngineer(benchmark_data=benchmark_data)
    strategy = SEPAStrategy(benchmark_data=benchmark_data)

    # Create trading config
    labeling_function = None
    if custom_label_rule_b.strip():
        labeling_function = eval(f"lambda trade: 1 if ({custom_label_rule_b}) else 0")

    trading_config = TradingConfig(
        success_threshold_pct=success_threshold_b,
        exit_on_trend_break=True,
        exit_on_stop_loss=False,
        allow_reentry=True,
        reentry_cooldown_days=0,
        labeling_function=labeling_function
    )

    # Initialize simulator
    if use_fast_simulator_b:
        from src.trade_simulator_fast import FastTradeSimulator
        simulator = FastTradeSimulator(
            data_repo=data_repo,
            strategy=strategy,
            feature_engine=feature_engine,
            start_date=start_date_b,
            end_date=end_date_b,
            outcome_end=outcome_end_b,
            config=trading_config
        )
        dataset_b = simulator.run_simulation(show_progress=True, n_jobs=n_jobs_b)
    else:
        from src.trade_simulator import TradeSimulator
        simulator = TradeSimulator(
            data_repo=data_repo,
            strategy=strategy,
            feature_engine=feature_engine,
            start_date=start_date_b,
            end_date=end_date_b,
            outcome_end=outcome_end_b,
            config=trading_config
        )
        dataset_b = simulator.run_simulation()

    if not dataset_b.empty:
        # Save dataset
        output_path = Path(output_path_b)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        dataset_b.to_parquet(output_path, index=False)

        # Get statistics
        stats = simulator.get_summary_statistics()

        # Display summary
        print(f"\n✅ Dataset B built successfully!")
        print(f"\n📊 Trade Statistics:")
        print(f"   Total Trades: {stats['total_trades']}")
        print(f"   Win Rate: {stats['win_rate']*100:.1f}%")
        print(f"   Avg Return: {stats['avg_return']:.2f}%")
        print(f"   Avg Days Held: {stats['avg_days_held']:.1f}")
        print(f"\n🏷️  Label Distribution:")
        for label, count in stats['label_distribution'].items():
            label_name = "Success" if label == 1 else "Failure"
            pct = (count / stats['total_trades']) * 100
            print(f"   {label_name}: {count} ({pct:.1f}%)")
        print(f"\n💾 File: {output_path}")
    else:
        print("❌ No trades generated!")

except Exception as e:
    print(f"❌ Error building Dataset B: {e}")
    import traceback
    traceback.print_exc()

## Cell 8: Merge and Prepare Training Data

Merge Dataset A + B and perform validation checks.

In [ ]:
# ============================================================================
# CELL 8: Merge and Prepare Training Data
# ============================================================================

print("🔗 DATASET MERGER - Prepare Final Training Data")
print("=" * 80)

# === CONFIGURATION ===
# Purpose: Merge Dataset A (features) + Dataset B (labels) and perform validation

# Input Paths
# Purpose: Specify where to load the datasets from
dataset_a_path_merge = 'data/ml/dataset_a.parquet'  # Example: 'data/ml/dataset_a.parquet'
dataset_b_path_merge = 'data/ml/dataset_b.parquet'  # Example: 'data/ml/dataset_b.parquet'

# Output Path
# Purpose: Where to save the merged training dataset
output_path_merge = 'data/ml/training_dataset_final.parquet'  # Example: 'data/ml/training_dataset_final.parquet'

# Validation
# Purpose: Perform temporal consistency checks to prevent data leakage
validate_temporal_merge = True  # Example: True (recommended), False (skip validation)

# Report Path
# Purpose: Where to save the validation report
report_path_merge = 'data/ml/preparation_report.txt'  # Example: 'data/ml/preparation_report.txt'

# === EXECUTION ===
training_dataset = None  # Global variable to store merged dataset

try:
    print(f"\n🔗 Merging datasets and validating...\n")

    # Import preparer
    from prepare_training_dataset import DatasetPreparer

    # Get date range from datasets
    # We'll use a dummy range for now (preparer will check actual range)
    preparer = DatasetPreparer(
        start_date='2020-01-01',
        end_date=datetime.now().strftime('%Y-%m-%d')
    )

    # Check Dataset A coverage
    print("[1/4] Checking Dataset A coverage...")
    dataset_a_stats = preparer.check_dataset_a_coverage(dataset_a_path_merge)
    preparer.validation_report['dataset_a'] = dataset_a_stats

    if not dataset_a_stats.get('exists'):
        print(f"❌ Dataset A not found: {dataset_a_path_merge}")
        raise FileNotFoundError(f"Dataset A not found: {dataset_a_path_merge}")
    print(f"   ✅ Found {dataset_a_stats['total_rows']:,} rows, {dataset_a_stats['total_features']} features")

    # Check Dataset B coverage
    print("\n[2/4] Checking Dataset B coverage...")
    dataset_b_stats = preparer.check_dataset_b_coverage(dataset_b_path_merge)
    preparer.validation_report['dataset_b'] = dataset_b_stats

    if not dataset_b_stats.get('exists'):
        print(f"❌ Dataset B not found: {dataset_b_path_merge}")
        raise FileNotFoundError(f"Dataset B not found: {dataset_b_path_merge}")
    print(f"   ✅ Found {dataset_b_stats['total_trades']:,} trades, {dataset_b_stats['win_rate']*100:.1f}% win rate")

    # Merge datasets
    print("\n[3/4] Merging datasets...")
    training_dataset = preparer.merge_datasets(
        dataset_a_path=dataset_a_path_merge,
        dataset_b_path=dataset_b_path_merge,
        output_path=output_path_merge
    )
    print(f"   ✅ Merged {len(training_dataset):,} rows")

    # Sanity checks
    print("\n[4/4] Running sanity checks...")
    sanity_results = preparer.perform_sanity_checks()
    preparer.validation_report['sanity_checks'] = sanity_results

    # Display results
    status_icon = {'PASS': '✅', 'WARNING': '⚠️', 'FAIL': '❌'}
    overall = sanity_results.get('overall_status', 'UNKNOWN')
    print(f"   {status_icon.get(overall, '❓')} Overall Status: {overall}")
    print(f"   Failures: {sanity_results.get('fail_count', 0)}")
    print(f"   Warnings: {sanity_results.get('warning_count', 0)}")

    # Generate and save report
    preparer.export_report(report_path_merge)

    # Final summary
    print(f"\n✅ Dataset preparation complete!")
    print(f"\n📊 Final Training Dataset:")
    print(f"   Rows: {len(training_dataset):,}")
    print(f"   Features: {len([c for c in training_dataset.columns if c not in ['date', 'ticker', 'entry_date', 'exit_date', 'label']])}")
    print(f"   Tickers: {training_dataset['ticker'].nunique()}")
    print(f"   File: {output_path_merge}")
    print(f"   Report: {report_path_merge}")

except Exception as e:
    print(f"❌ Error merging datasets: {e}")
    import traceback
    traceback.print_exc()

## Cell 9: Data Preparation / Feature Engineering Framework

Template for custom data cleaning and feature engineering.

In [ ]:
# ============================================================================
# CELL 9: Data Preparation / Feature Engineering Framework
# ============================================================================

print("🔧 FEATURE ENGINEERING FRAMEWORK")
print("=" * 80)
print("\nThis cell provides a framework for custom feature engineering.")
print("Modify the code below to add your custom logic.\n")

# === CONFIGURATION ===
training_dataset_path_fe = 'data/ml/training_dataset_final.parquet'
output_path_fe = 'data/ml/training_dataset_cleaned.parquet'

# === LOAD DATASET ===
print("[1/5] Loading training dataset...")
df_fe = pd.read_parquet(training_dataset_path_fe)
print(f"   Loaded {len(df_fe):,} rows, {len(df_fe.columns)} columns\n")

# === HANDLE MISSING VALUES ===
print("[2/5] Handling missing values...")
print(f"   Missing values before: {df_fe.isnull().sum().sum():,} ({df_fe.isnull().sum().sum()/df_fe.size*100:.2f}%)")

# TODO: Add your custom missing value handling here
# Example:
# df_fe['feature_name'] = df_fe['feature_name'].fillna(df_fe['feature_name'].median())

print(f"   Missing values after: {df_fe.isnull().sum().sum():,} ({df_fe.isnull().sum().sum()/df_fe.size*100:.2f}%)\n")

# === FEATURE SCALING / NORMALIZATION ===
print("[3/5] Feature scaling / normalization...")

# TODO: Add your custom scaling logic here
# Example:
# from sklearn.preprocessing import StandardScaler
# scaler = StandardScaler()
# df_fe[numeric_cols] = scaler.fit_transform(df_fe[numeric_cols])

print("   Scaling complete\n")

# === FEATURE CREATION ===
print("[4/5] Creating custom features...")

# TODO: Add your custom feature creation here
# Example:
# df_fe['custom_feature'] = df_fe['feature1'] / df_fe['feature2']

print(f"   Total features: {len(df_fe.columns)}\n")

# === SAVE CLEANED DATASET ===
print("[5/5] Saving cleaned dataset...")
output_path = Path(output_path_fe)
output_path.parent.mkdir(parents=True, exist_ok=True)
df_fe.to_parquet(output_path, index=False)
print(f"   ✅ Saved to: {output_path}")
print(f"   Size: {output_path.stat().st_size / (1024*1024):.2f} MB\n")

print("✅ Feature engineering complete!")

## Cell 10: Define Fold and Training Parameters

Configure temporal splitting and feature selection parameters.

In [ ]:
# ============================================================================
# CELL 10: Define Fold and Training Parameters
# ============================================================================

print("⚙️ TRAINING CONFIGURATION")
print("=" * 80)

# === CONFIGURATION ===
# Purpose: Configure temporal cross-validation and feature selection for ML training

# Dataset Path
# Purpose: Location of the merged training dataset
dataset_path_train = 'data/ml/training_dataset_final.parquet'  # Example: 'data/ml/training_dataset_final.parquet'

# Temporal Splitting
# Purpose: Days to exclude between train and test to prevent look-ahead bias
# Larger values = more conservative but less training data
purge_gap_days = 60  # Example: 60 (recommended 30-120 days)

# Feature Selection
# Purpose: Remove highly correlated features to reduce overfitting
correlation_threshold = 0.95  # Example: 0.95 (remove features with >95% correlation)

# Top N Features
# Purpose: Keep only the N most important features (0 = keep all after correlation filtering)
top_n_features = 0  # Example: 0 (keep all), 50 (keep top 50 features)

# Evaluation Metric
# Purpose: Precision@k measures accuracy in the top k% of predictions
# This aligns with real trading (we only trade the highest-confidence signals)
precision_k_pct = 0.2  # Example: 0.2 (top 20% of predictions)

# Hyperparameter Optimization
# Purpose: Use Optuna to find optimal XGBoost parameters (slow but improves accuracy)
optimize_hyperparameters = False  # Example: False (use defaults), True (run optimization)

# Optuna Trials
# Purpose: How many hyperparameter combinations to try (only used if optimize_hyperparameters=True)
n_optuna_trials = 50  # Example: 50 (10-100 recommended)

# === EXECUTION ===
train_config = {}  # Global variable to store configuration

try:
    print(f"\n🔧 Preparing training configuration...\n")

    # Import preparation modules
    from src.model_preparation import prepare_training_data

    # Prepare data
    top_n = top_n_features if top_n_features > 0 else None

    prep_result = prepare_training_data(
        dataset_path=dataset_path_train,
        purge_gap_days=purge_gap_days,
        correlation_threshold=correlation_threshold,
        keep_top_n=top_n
    )

    # Store configuration
    train_config = {
        'df': prep_result['df'],
        'folds': prep_result['folds'],
        'splitter': prep_result['splitter'],
        'selector': prep_result['selector'],
        'optimize': optimize_hyperparameters,
        'n_trials': n_optuna_trials,
        'precision_k': precision_k_pct
    }

    # Display summary
    print(f"✅ Training preparation complete!\n")
    print(f"📊 Configuration:")
    print(f"   Dataset: {len(train_config['df']):,} rows")
    print(f"   Temporal Folds: {len(train_config['folds'])}")
    print(f"   Selected Features: {len(train_config['selector'].selected_features)}")
    print(f"   Dropped Features: {sum(len(v) for v in train_config['selector'].dropped_features.values())}")
    print(f"   Purge Gap: {purge_gap_days} days")
    print(f"   Optimize Hyperparameters: {optimize_hyperparameters}")
    if optimize_hyperparameters:
        print(f"   Optuna Trials: {n_optuna_trials}")
    print(f"   Precision@k: {int(precision_k_pct*100)}%")

    # Display fold structure
    print(f"\n📅 Fold Structure:")
    for fold in train_config['folds']:
        fold_type = "PRODUCTION" if fold.get('is_production', False) else "Validation"
        print(f"   Fold {fold['fold_id']} ({fold_type}):")
        print(f"     Train: {fold['train_start']} to {fold['train_end']} ({fold['train_size']:,} rows)")
        if not fold.get('is_production', False):
            print(f"     Test:  {fold['test_start']} to {fold['test_end']} ({fold['test_size']:,} rows)")

except Exception as e:
    print(f"❌ Error preparing training: {e}")
    import traceback
    traceback.print_exc()

## Cell 11: Train Model

Train XGBoost models for each temporal fold.

In [ ]:
# ============================================================================
# CELL 11: Train Model
# ============================================================================

print("🚀 MODEL TRAINING")
print("=" * 80)

# === CONFIGURATION ===
# Purpose: Train XGBoost models for each temporal fold

# Model Output Directory
# Purpose: Where to save trained model files
model_output_dir = 'models'  # Example: 'models'

# Evaluation Output Directory
# Purpose: Where to save evaluation plots and metrics
eval_output_dir = 'evaluation'  # Example: 'evaluation'

# GPU Acceleration
# Purpose: Use GPU for faster training (requires XGBoost with CUDA support)
use_gpu = False  # Example: False (CPU), True (GPU if available)

# === EXECUTION ===
trained_models = []  # Global variable to store trained models

try:
    # Check if training config exists
    if not train_config:
        print("❌ No training configuration! Please run Cell 10 first.")
        raise ValueError("Training configuration not found")

    print(f"\n🚀 Training {len(train_config['folds'])} fold(s)...\n")

    # Import trainer
    from src.train_model import SEPAModelTrainer

    # Create output directories
    Path(model_output_dir).mkdir(parents=True, exist_ok=True)
    Path(eval_output_dir).mkdir(parents=True, exist_ok=True)

    trained_models = []

    for fold_idx, fold in enumerate(train_config['folds']):
        fold_type = "PRODUCTION" if fold.get('is_production', False) else "Validation"
        print(f"\n{'='*80}")
        print(f"TRAINING FOLD {fold['fold_id']} ({fold_type})")
        print(f"{'='*80}\n")

        # Get train/test data
        X_train, X_test, y_train, y_test = train_config['splitter'].get_fold_data(
            train_config['df'],
            fold_idx=fold_idx,
            feature_columns=train_config['selector'].selected_features
        )

        # Apply feature selection
        X_train = train_config['selector'].transform(X_train)
        X_test = train_config['selector'].transform(X_test)

        # Split train/val for validation folds
        if fold.get('is_production', False):
            X_train_split = X_train
            y_train_split = y_train
            X_val_split = None
            y_val_split = None
            print(f"Production fold: Using ALL training data ({len(X_train_split)} rows)\n")
        else:
            train_size = int(len(X_train) * 0.8)
            X_train_split = X_train.iloc[:train_size]
            y_train_split = y_train.iloc[:train_size]
            X_val_split = X_train.iloc[train_size:]
            y_val_split = y_train.iloc[train_size:]
            print(f"Train: {len(X_train_split)}, Val: {len(X_val_split)}, Test: {len(X_test)} rows\n")

        # Initialize trainer
        trainer = SEPAModelTrainer(precision_k_pct=train_config['precision_k'])

        # Optimize hyperparameters (only on first fold)
        if train_config['optimize'] and fold_idx == 0:
            print(f"Optimizing hyperparameters ({train_config['n_trials']} trials)...")
            best_params = trainer.optimize_hyperparameters(
                X_train_split,
                y_train_split,
                X_val_split,
                y_val_split,
                n_trials=train_config['n_trials']
            )
            print(f"Best parameters found!\n")
        elif fold_idx > 0 and train_config['optimize']:
            print("Reusing hyperparameters from Fold 1\n")
            trainer.best_params = trained_models[0].best_params

        # Train model
        print("Training final model...")
        model = trainer.train_with_best_params(
            X_train_split,
            y_train_split,
            X_val_split,
            y_val_split
        )

        # Save model
        trainer.save_model(model_output_dir, fold_id=fold['fold_id'])
        trained_models.append(trainer)

        print(f"\n✅ Fold {fold['fold_id']} training complete!")

    print(f"\n{'='*80}")
    print(f"✅ ALL MODELS TRAINED SUCCESSFULLY!")
    print(f"{'='*80}")
    print(f"\n📁 Models saved to: {Path(model_output_dir).resolve()}")
    print(f"   Total models: {len(trained_models)}")

except Exception as e:
    print(f"❌ Error training models: {e}")
    import traceback
    traceback.print_exc()

## Cell 12: Review Results and Select Best Model

Compare fold performance and select the best model for production.

In [ ]:
# ============================================================================
# CELL 12: Review Results and Select Best Model
# ============================================================================

print("📊 MODEL EVALUATION & SELECTION")
print("=" * 80)
print("\nThis cell evaluates all trained folds and helps you select the best model.\n")

# Check if models are trained
if not trained_models:
    print("❌ No trained models found! Please run Cell 11 first.")
else:
    # Import evaluator
    from src.evaluate_model import ModelEvaluator
    
    eval_dir = eval_output_dir.value
    evaluator = ModelEvaluator(output_dir=eval_dir)
    
    # Evaluate each fold (skip production folds)
    all_fold_results = []
    
    for fold_idx, fold in enumerate(train_config['folds']):
        if fold.get('is_production', False):
            print(f"Skipping Fold {fold['fold_id']} (PRODUCTION - no test data)\n")
            continue
        
        print(f"Evaluating Fold {fold['fold_id']}...")
        
        # Get test data
        X_train, X_test, y_train, y_test = train_config['splitter'].get_fold_data(
            train_config['df'],
            fold_idx=fold_idx,
            feature_columns=train_config['selector'].selected_features
        )
        
        X_test = train_config['selector'].transform(X_test)
        test_indices = fold['test_indices']
        df_test = train_config['df'].loc[test_indices]
        
        # Get trained model
        model = trained_models[fold_idx].model
        
        # Evaluate
        fold_results = evaluator.evaluate_fold(
            model=model,
            X_test=X_test,
            y_test=y_test,
            df_test=df_test,
            fold_id=fold['fold_id'],
            k_values=[0.1, 0.2, 0.3]
        )
        
        all_fold_results.append(fold_results)
    
    # Generate comprehensive report
    evaluator.generate_report(
        all_fold_results,
        output_path=f'{eval_dir}/evaluation_report.json'
    )
    
    # Create comparison table
    comparison_data = []
    for result in all_fold_results:
        comparison_data.append({
            'Fold': result['fold_id'],
            'Precision@10%': result['metrics']['precision_at_k'].get(0.1, 0),
            'Precision@20%': result['metrics']['precision_at_k'].get(0.2, 0),
            'Precision@30%': result['metrics']['precision_at_k'].get(0.3, 0),
            'Win Rate': result['backtest']['win_rate'],
            'Avg Return': result['backtest']['avg_return'],
            'AUC-ROC': result['metrics']['auc_roc']
        })
    
    comparison_df = pd.DataFrame(comparison_data)
    
    print(f"\n✅ Evaluation complete!\n")
    print("📊 FOLD COMPARISON:")
    print("=" * 80)
    display(comparison_df)
    
    # Visualizations
    print(f"\n📈 Visualizations saved to: {eval_dir}/")
    print(f"   - ROC curves: roc_curve_fold_*.png")
    print(f"   - Feature importance: feature_importance_fold_*.png")
    print(f"   - Full report: evaluation_report.json")
    
    # Selection helper
    print(f"\n🎯 MODEL SELECTION HELPER:")
    print("=" * 80)
    
    best_by_precision = comparison_df.loc[comparison_df['Precision@20%'].idxmax()]
    best_by_return = comparison_df.loc[comparison_df['Avg Return'].idxmax()]
    
    print(f"\n   Best by Precision@20%: Fold {best_by_precision['Fold']} ({best_by_precision['Precision@20%']:.3f})")
    print(f"   Best by Avg Return: Fold {best_by_return['Fold']} ({best_by_return['Avg Return']:.2f}%)")
    
    print(f"\n💡 Recommendation: Review the evaluation report and plots before selecting.")
    print(f"   Consider consistency across folds and alignment with your strategy goals.\n")

## Cell 13: Retrain Production Model

Retrain selected model on ALL available data for production deployment.

In [ ]:
# ============================================================================
# CELL 13: Retrain Production Model
# ============================================================================

print("🏭 PRODUCTION MODEL TRAINING")
print("=" * 80)

# === CONFIGURATION ===
# Purpose: Retrain the best model on ALL available data for production deployment

# Best Fold Selection
# Purpose: Which fold's hyperparameters to use (based on Cell 12 evaluation results)
best_fold_id = 1  # Example: 1 (use hyperparameters from Fold 1)

# Training Data
# Purpose: Use all available data (no holdout) to maximize model performance
use_all_data_prod = True  # Example: True (recommended for production)

# Production Model Path
# Purpose: Where to save the final production model
production_model_path = 'models/model_production.json'  # Example: 'models/model_production.json'

# === EXECUTION ===
production_model = None  # Global variable

try:
    # Check if models exist
    if not trained_models:
        print("❌ No trained models! Please run Cell 11 first.")
        raise ValueError("No trained models available")

    print(f"\n🏭 Training production model...\n")
    print(f"   Using hyperparameters from Fold {best_fold_id}")
    print(f"   Training on ALL data: {use_all_data_prod}\n")

    # Get best model's hyperparameters
    fold_idx = best_fold_id - 1  # 0-indexed
    best_trainer = trained_models[fold_idx]

    # Import trainer
    from src.train_model import SEPAModelTrainer

    # Initialize new trainer with best params
    prod_trainer = SEPAModelTrainer(precision_k_pct=train_config['precision_k'])
    prod_trainer.best_params = best_trainer.best_params

    # Prepare data
    if use_all_data_prod:
        # Use ALL data (no split)
        feature_cols = train_config['selector'].selected_features
        X_all = train_config['df'][feature_cols]
        y_all = train_config['df']['label']

        # Apply feature selection
        X_all = train_config['selector'].transform(X_all)

        print(f"   Training on {len(X_all):,} total rows\n")

        # Train
        production_model = prod_trainer.train_with_best_params(
            X_all,
            y_all,
            None,  # No validation data
            None
        )
    else:
        print("❌ Non-all-data training not implemented. Use all data for production.")
        raise ValueError("use_all_data_prod must be True")

    # Save production model
    prod_path = Path(production_model_path)
    prod_path.parent.mkdir(parents=True, exist_ok=True)
    production_model.save_model(str(prod_path))

    print(f"\n✅ Production model trained successfully!\n")
    print(f"📊 Model Info:")
    print(f"   Features: {len(train_config['selector'].selected_features)}")
    print(f"   Training Rows: {len(X_all):,}")
    print(f"   Hyperparameters: Inherited from Fold {best_fold_id}")
    print(f"   Model Path: {prod_path}")
    print(f"   Model Version: {datetime.now().strftime('%Y%m%d_%H%M%S')}")

    print(f"\n🚀 Ready for deployment in scanner (use --use-ml flag)!\n")

except Exception as e:
    print(f"❌ Error training production model: {e}")
    import traceback
    traceback.print_exc()

---
# SECTION 3: DAILY SCANNER
---

## Cell 14: Define Scanner Scope

Configure and run the daily scanner (single day or date range).

In [ ]:
# ============================================================================
# CELL 14: Define Scanner Scope
# ============================================================================

print("🔍 DAILY SCANNER")
print("=" * 80)

# === CONFIGURATION ===
# Purpose: Run the daily scanner to find buy/sell signals

# Scan Mode
# Purpose: Choose between single-day or date-range scanning
# Options: 'single_day' (one specific date), 'date_range' (scan multiple days)
scan_mode = 'single_day'  # Example: 'single_day', 'date_range'

# Single Day Scan Date
# Purpose: Which date to scan (only used if scan_mode='single_day')
scan_date_single = datetime.now().strftime('%Y-%m-%d')  # Example: datetime.now() (today)

# Date Range (only used if scan_mode='date_range')
# Purpose: Define start and end dates for multi-day scanning
start_date_range = (datetime.now() - timedelta(days=30)).strftime('%Y-%m-%d')  # Example: 30 days ago
end_date_range = datetime.now().strftime('%Y-%m-%d')  # Example: today

# ML Scoring
# Purpose: Enable ML model to rank and filter signals
use_ml_scanner = False  # Example: False (rule-based only), True (use ML scoring)

# ML Model Path (only used if use_ml_scanner=True)
# Purpose: Path to production model file
model_path_scanner = 'models/model_production.json'  # Example: 'models/model_production.json'

# Output Options
# Purpose: Export results to CSV file
csv_output_scanner = False  # Example: False (display only), True (save to CSV)

# Debug Mode
# Purpose: Show detailed metrics for debugging
debug_mode_scanner = False  # Example: False (summary only), True (detailed metrics)

# Ticker Filter (optional)
# Purpose: Scan only specific tickers instead of full universe
# Leave empty to scan all tickers
scanner_tickers_input = ''  # Example: '', 'AAPL,MSFT,TSLA'

# === EXECUTION ===
try:
    # Parse tickers
    scanner_tickers = None
    if scanner_tickers_input.strip():
        scanner_tickers = [t.strip().upper() for t in scanner_tickers_input.split(',') if t.strip()]

    # Import scanner
    from optimized_scanner import run_optimized_scanner

    if scan_mode == 'single_day':
        print(f"\n🔍 Running single-day scan for {scan_date_single}...\n")

        run_optimized_scanner(
            scan_date=scan_date_single,
            csv_output=csv_output_scanner,
            use_ml=use_ml_scanner,
            model_path=model_path_scanner if use_ml_scanner else None,
            tickers=scanner_tickers,
            debug=debug_mode_scanner
        )
    else:
        print(f"\n🔍 Running date-range scan from {start_date_range} to {end_date_range}...\n")
        print("⚠️  Date range mode uses 2D vectorized scanning (faster)\n")

        # Import components for date range mode
        import sys
        from datetime import datetime, timedelta
        import pandas as pd
        import time

        # Simulate command-line args
        class Args:
            def __init__(self):
                self.date_range = [start_date_range, end_date_range]
                self.use_ml = use_ml_scanner
                self.model_path = model_path_scanner if use_ml_scanner else None

        args = Args()

        # Run date range scan
        from src.data_engine import DataRepository
        from src.features import FeatureEngineer
        from src.vectorized_screening import VectorizedSEPAScreener

        start_date = datetime.strptime(args.date_range[0], '%Y-%m-%d')
        end_date = datetime.strptime(args.date_range[1], '%Y-%m-%d')

        print("[PRE-SCAN] Updating cache...")
        data_repo = DataRepository()
        tickers = data_repo.update_universe()
        min_date = (pd.Timestamp.now() - pd.DateOffset(years=2)).strftime('%Y-%m-%d')
        results = data_repo.update_cache(tickers, force=False, source='fmp', min_date=min_date)

        print("[PRE-SCAN] Loading price data...")
        all_ticker_data = data_repo.get_batch_data(tickers)

        print("\n[Step 1/4] Computing features...")
        benchmark_data = data_repo.get_benchmark_data()
        feature_engine = FeatureEngineer(benchmark_data=benchmark_data)
        enriched_data = feature_engine.process_universe_batch(all_ticker_data)

        print("[Step 2/4] Building 2D matrix...")
        lookback_days = 251
        data_start_date = start_date - timedelta(days=lookback_days)
        df_matrix = VectorizedSEPAScreener.build_2d_matrix(
            enriched_data,
            start_date=pd.Timestamp(data_start_date),
            end_date=pd.Timestamp(end_date)
        )

        print("[Step 3/4] Computing SEPA status...")
        df_matrix = VectorizedSEPAScreener.add_sepa_status_column(df_matrix)

        print("[Step 4/4] Finding signal transitions...")
        buy_signals, sell_signals = VectorizedSEPAScreener.find_signal_transitions(
            df_matrix,
            date_range_start=pd.Timestamp(start_date),
            date_range_end=pd.Timestamp(end_date)
        )

        print(f"\n✅ Date range scan complete!")
        print(f"   Buy signals: {len(buy_signals)}")
        print(f"   Sell signals: {len(sell_signals)}")

except Exception as e:
    print(f"❌ Error running scanner: {e}")
    import traceback
    traceback.print_exc()

## Cell 15: Review Buy List

Display and analyze the current buy list from the database.

In [ ]:
# ============================================================================
# CELL 15: Review Buy List
# ============================================================================

print("📋 BUY LIST VIEWER")
print("=" * 80)

# === CONFIGURATION ===
# Purpose: Display and analyze the current buy list from the database

# As Of Date
# Purpose: View buy list as of a specific date (for historical analysis)
as_of_date_buylist = datetime.now().strftime('%Y-%m-%d')  # Example: datetime.now() (current state)

# Active Only
# Purpose: Show only active signals or include closed/exited signals
active_only_buylist = True  # Example: True (active only), False (all signals)

# Sort By
# Purpose: How to order the results
# Options: 'signal_date', 'ml_rank', 'rs', 'price_change_%'
sort_by_buylist = 'ml_rank'  # Example: 'ml_rank', 'signal_date', 'rs'

# Export to CSV
# Purpose: Save buy list to file
export_csv_buylist = False  # Example: False (display only), True (export to CSV)

# === EXECUTION ===
try:
    print(f"\n📋 Loading buy list...\n")

    # Initialize database
    db = DatabaseManager()

    # Get buy list
    buy_list_df = db.get_buy_list(
        active_only=active_only_buylist,
        as_of_date=as_of_date_buylist
    )

    if buy_list_df.empty:
        print("No active buy signals found.")
    else:
        # Calculate price changes
        if 'signal_price' in buy_list_df.columns and 'current_price' in buy_list_df.columns:
            buy_list_df['price_change_$'] = buy_list_df['current_price'] - buy_list_df['signal_price']
            buy_list_df['price_change_%'] = ((buy_list_df['current_price'] - buy_list_df['signal_price']) / buy_list_df['signal_price'] * 100)

        # Sort
        if sort_by_buylist in buy_list_df.columns:
            buy_list_df = buy_list_df.sort_values(sort_by_buylist)

        # Display summary
        print(f"✅ Buy List loaded ({len(buy_list_df)} signals)\n")
        print(f"📊 Summary Statistics:")

        if 'signal_date' in buy_list_df.columns:
            buy_list_df['days_on_list'] = (pd.Timestamp(as_of_date_buylist) - pd.to_datetime(buy_list_df['signal_date'])).dt.days
            print(f"   Avg days on list: {buy_list_df['days_on_list'].mean():.1f}")
            print(f"   Newest signal: {buy_list_df['signal_date'].max()}")
            print(f"   Oldest signal: {buy_list_df['signal_date'].min()}")

        if 'price_change_%' in buy_list_df.columns:
            print(f"\n💰 Performance:")
            print(f"   Avg return: {buy_list_df['price_change_%'].mean():.2f}%")
            print(f"   Best performer: {buy_list_df.loc[buy_list_df['price_change_%'].idxmax(), 'ticker']} (+{buy_list_df['price_change_%'].max():.2f}%)")
            print(f"   Worst performer: {buy_list_df.loc[buy_list_df['price_change_%'].idxmin(), 'ticker']} ({buy_list_df['price_change_%'].min():.2f}%)")

        # Display dataframe
        print(f"\n📋 BUY LIST:\n")
        display_cols = ['ticker', 'signal_date', 'signal_price', 'current_price', 'price_change_%',
                       'rs', 'volume_ratio', 'ml_probability', 'ml_rank']
        available_cols = [col for col in display_cols if col in buy_list_df.columns]
        display(buy_list_df[available_cols].head(50))

        # Export if requested
        if export_csv_buylist:
            output_path = Path('data/scanner_output') / f'buy_list_{as_of_date_buylist}.csv'
            output_path.parent.mkdir(parents=True, exist_ok=True)
            buy_list_df.to_csv(output_path, index=False)
            print(f"\n💾 Exported to: {output_path}")

except Exception as e:
    print(f"❌ Error loading buy list: {e}")
    import traceback
    traceback.print_exc()

## Cell 16: Database Management Examples

Demonstrate common database operations and queries.

In [ ]:
# ============================================================================
# CELL 16: Database Management Examples
# ============================================================================

print("🗄️ DATABASE MANAGEMENT")
print("=" * 80)
print("\nThis cell provides examples of common database operations.\n")

# Initialize database
db = DatabaseManager()

print("Available operations:")
print("  1. Inspect buy_list table")
print("  2. Inspect buy_list_activity log")
print("  3. Clear future signals (temporal consistency)")
print("  4. Export tables to CSV")
print("  5. Custom SQL query\n")

# === OPERATION 1: Inspect buy_list ===
print("[1] BUY LIST TABLE:")
print("=" * 80)
buy_list = db.get_buy_list(active_only=False)
print(f"Total entries: {len(buy_list)}")
print(f"Active: {len(buy_list[buy_list['is_active'] == 1]) if 'is_active' in buy_list.columns else 'N/A'}")
if not buy_list.empty:
    display(buy_list.head(10))
else:
    print("(Empty table)")

# === OPERATION 2: Inspect activity log ===
print("\n[2] BUY LIST ACTIVITY LOG:")
print("=" * 80)
# Export to temporary file to view
temp_activity_path = 'data/temp_activity.csv'
db.export_to_csv('buy_list_activity', temp_activity_path)
activity_df = pd.read_csv(temp_activity_path)
print(f"Total activity records: {len(activity_df)}")
if not activity_df.empty:
    print(f"\nRecent activity:")
    display(activity_df.tail(10))
else:
    print("(No activity logged)")

# === OPERATION 3: Clear future signals (example) ===
print("\n[3] CLEAR FUTURE SIGNALS (Example - NOT EXECUTED):")
print("=" * 80)
print("To clear signals after a specific date:")
print("  deleted = db.clear_future_signals('2024-01-01')")
print("  print(f'Cleared {deleted[\"buy_list_deleted\"]} signals')")

# === OPERATION 4: Export tables ===
print("\n[4] EXPORT TABLES TO CSV:")
print("=" * 80)
export_dir = Path('data/db_exports')
export_dir.mkdir(parents=True, exist_ok=True)

tables_to_export = ['buy_list', 'buy_list_activity']
for table in tables_to_export:
    export_path = export_dir / f'{table}_{datetime.now().strftime("%Y%m%d")}.csv'
    try:
        db.export_to_csv(table, str(export_path))
        print(f"  ✅ {table}: {export_path}")
    except Exception as e:
        print(f"  ❌ {table}: {e}")

# === OPERATION 5: Custom SQL query (example) ===
print("\n[5] CUSTOM SQL QUERY (Example):")
print("=" * 80)
print("Example query to find top ML-ranked stocks:")
print("")
print("SELECT ticker, ml_rank, ml_probability, signal_date")
print("FROM buy_list")
print("WHERE is_active = 1 AND ml_rank IS NOT NULL")
print("ORDER BY ml_rank ASC")
print("LIMIT 10;")
print("")
print("To execute:")
print("  result = db.execute_query('SELECT ...')")

print("\n✅ Database management operations complete!")